<a href="https://colab.research.google.com/github/jonathanwn-23/Praktik-LLM-Gemini-API/blob/main/Praktik_LLM_Gemini_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktik LLM dengan API Gemini

Notebook ini membahas cara memanggil model bahasa besar (LLM) lewat API Gemini, lalu mengamati bagaimana parameter dan cara penulisan prompt memengaruhi hasilnya.

**Yang akan kamu lakukan:**
1. Menyiapkan API key dan memanggil model pertama kali.
2. Membaca jumlah token dari setiap respons.
3. Mengamati pengaruh `temperature`.
4. Mengatur output agar berbentuk JSON dan memeriksanya.
5. Mengirim riwayat percakapan agar model terlihat "mengingat".
6. Membandingkan hasil beberapa prompt dengan kriteria yang sama.

**Cara menjalankan sel:** klik ▶ di sebelah kiri kotak kode, atau tekan Shift + Enter.

**Yang perlu disiapkan:** API key dari Google AI Studio (https://aistudio.google.com/apikey), disimpan di panel **Secrets** (ikon kunci) dengan nama `GEMINI_API_KEY` dan Notebook access aktif. Jangan menulis key langsung di kode.

## Langkah awal: membuat API key dan menyimpannya di Colab

Kerjakan langkah ini sebelum menjalankan sel kode apa pun. Tanpa API key, semua sel akan gagal.

### Langkah 1. Membuat API key di Google AI Studio
1. Buka https://aistudio.google.com/apikey lalu login dengan akun Google kamu.
2. Klik **Create API key**. Pilih project yang sudah ada, atau biarkan project dibuat otomatis.
3. Salin key yang muncul. Simpan sementara di tempat aman sampai langkah 3 selesai.

> **Screenshot S01:** halaman API key dengan key yang sudah disensor.

### Langkah 2. Membuka notebook di Colab
Notebook ini sudah berjalan di Colab. Jika file `.ipynb` dibuka dari luar Colab, unggah dulu lewat https://colab.research.google.com dengan memilih **File**, lalu **Upload notebook**.

> **Screenshot S02:** judul notebook dan panel kiri Colab.

### Langkah 3. Menyimpan key di Secrets
1. Di panel kiri Colab, klik ikon **kunci** (**Secrets**). Panel Secrets akan terbuka.
2. Pilih salah satu cara:
   - **Cara A (paling mudah):** klik **Add new secret**, lalu pilih **Import key from Google AI Studio**. Pilih key yang kamu buat di langkah 1. Periksa nama secret-nya, harus tepat `GEMINI_API_KEY`. Jika namanya berbeda, ubah dulu.
   - **Cara B (manual):** di baris kosong, isi kolom **Name** dengan `GEMINI_API_KEY` (huruf besar semua, tanpa spasi). Di kolom **Value**, tempel key dari langkah 1.
3. Pastikan tombol **Notebook access** aktif. Tombolnya harus biru dengan tanda centang. Jika masih abu-abu atau bertanda silang, klik sampai aktif.
4. Key tersimpan di Secrets, sehingga tidak perlu ditulis di dalam kode.

> **Screenshot S03:** panel Secrets dengan nama `GEMINI_API_KEY` dan Notebook access aktif. Nilai key harus disensor.

### Langkah 4. Menjalankan sel instalasi dan koneksi
Lanjut ke bagian **1. Instalasi dan koneksi** di bawah. Jalankan sel-selnya secara berurutan dengan tombol ▶.

**Jika muncul pesan minta izin** untuk mengakses secret, klik **Allow**.

**Jika ada error:**
- **SecretNotFoundError:** nama secret salah atau belum dibuat. Periksa ejaannya, harus `GEMINI_API_KEY`.
- **Error soal akses secret:** Notebook access belum aktif. Nyalakan lagi di panel Secrets, lalu jalankan ulang sel.
- **Key tidak berfungsi:** pastikan key tersalin utuh tanpa spasi di awal atau akhir. Jika ragu, buat key baru di AI Studio dan simpan ulang di Secrets.

> **Screenshot S04:** hasil sel daftar model dengan nama model yang tampil.

**Keamanan:**
- Jangan menempel key di kode, di chat, atau di screenshot yang dibagikan.
- Jika key terbagikan secara tidak sengaja, hapus di halaman AI Studio yang sama, lalu buat yang baru.
- Setelah praktik selesai, hapus key di AI Studio jika tidak akan dipakai lagi.

## Generative AI tidak hanya LLM

LLM adalah salah satu jenis generative AI, yaitu yang fokus pada bahasa. Generative AI secara umum adalah model yang menghasilkan konten baru, dan konten itu bisa berupa gambar, suara, video, atau kode.

| Jenis | Yang dihasilkan | Prinsip singkat | Contoh model atau produk | Contoh penggunaan di D3 TI |
|---|---|---|---|---|
| LLM | Teks dan kode | Memprediksi token berikutnya | Gemini, Claude, GPT, Llama | Chatbot, ringkasan, asisten kode |
| Image generation | Gambar | Diffusion: mengubah noise menjadi gambar mengikuti deskripsi teks | DALL·E, Midjourney, Stable Diffusion, Imagen | Membuat mockup antarmuka, ilustrasi poster |
| Audio dan speech | Suara dan musik | Model generatif untuk suara, sering dipakai bersama teks | Text-to-speech, layanan musik berbasis AI | Suara untuk aplikasi mobile, asisten suara |
| Video | Video pendek | Diffusion untuk urutan frame yang konsisten | Sora, Veo | Pembuatan materi pembelajaran |
| 3D dan desain | Objek dan model 3D | Model generatif untuk bentuk dan tekstur | Berbagai alat riset dan komersial | Prototipe objek untuk IoT |
| Model embedding | Vektor angka, bukan konten | Mengubah teks atau gambar menjadi vektor makna | Model embedding dari berbagai penyedia | Pencarian semantik dan RAG |

**Catatan:**
- Model embedding tidak menghasilkan konten baru, tetapi sering dianggap bagian dari ekosistem generative AI karena dipakai bersama LLM.
- Sebelum diffusion, ada GAN (Generative Adversarial Network) yang juga dikenal untuk menghasilkan gambar. Untuk mahasiswa, cukup tahu bahwa diffusion adalah pendekatan yang paling umum saat ini untuk gambar dan video.
- Nama produk berubah cepat. Tabel ini memberikan contoh, bukan daftar resmi.

**Pertanyaan diskusi:** untuk aplikasi web, mobile, atau IoT yang kamu pilih, jenis generative AI mana yang paling tepat, dan apakah cukup dengan LLM saja?

## Konsep inti LLM

Baca daftar ini dulu. Istilah-istilah di bawah akan muncul di latihan.

**Dasar**
- **LLM (Large Language Model):** model besar yang fokus memproses dan menghasilkan bahasa.
- **Foundation model:** model besar yang dilatih pada data luas, lalu menjadi dasar berbagai tugas.
- **Multimodal:** model yang bisa menerima lebih dari satu jenis input, misalnya teks dan gambar.

**Cara kerja**
- **Token:** potongan teks yang dibaca dan ditulis model. Jumlahnya tidak sama dengan jumlah kata.
- **Tokenizer:** aturan yang mengubah teks menjadi token.
- **Embedding:** representasi angka dari setiap token, yang mewakili makna awalnya.
- **Attention:** mekanisme yang membuat setiap token menimbang token lain di sekitarnya.
- **Transformer:** arsitektur model yang menumpuk banyak lapisan attention.
- **Next-token prediction:** cara model menghasilkan teks, yaitu menebak token berikutnya satu per satu.
- **Context window:** batas jumlah token yang bisa diproses dalam satu permintaan.
- **Konteks:** semua token yang dikirim dalam satu permintaan, termasuk instruksi, riwayat, dan dokumen.

**Siklus hidup model**
- **Pretraining:** pelatihan awal untuk mempelajari pola bahasa dari korpus yang sangat besar.
- **Instruction tuning:** pelatihan lanjutan agar model lebih baik mengikuti instruksi.
- **Alignment:** penyesuaian agar perilaku model sesuai kebijakan dan preferensi yang diinginkan.
- **Fine-tuning:** penyesuaian tambahan untuk perilaku atau kemampuan tertentu.
- **Inference:** fase ketika model dipakai aplikasi, dan bobotnya biasanya tidak berubah.

**Parameter generasi**
- **Temperature:** mengatur seberapa sering model memilih kandidat yang kurang umum.
- **Top-p:** membatasi kandidat token berdasarkan probabilitas kumulatif.
- **Max output tokens:** batas panjang jawaban.
- **Stop condition:** aturan kapan generasi berhenti.
- **System instruction:** instruksi yang mengatur peran dan batasan model.

**Risiko dan evaluasi**
- **Hallucination:** informasi yang tampak masuk akal tetapi tidak didukung fakta atau sumber.
- **Grounding:** jawaban yang didasarkan pada sumber yang bisa diperiksa.
- **Bias:** output yang tidak adil untuk kelompok tertentu, akibat data atau konteks penggunaan.
- **Prompt injection:** input yang disusun untuk membuat model mengabaikan instruksi aslinya.
- **Latency:** waktu tunggu sampai respons diterima.
- **Cost:** biaya yang dihitung dari jumlah token input dan output.

**Di luar praktik ini**
- **RAG (Retrieval-Augmented Generation):** mengambil dokumen yang relevan lalu memasukkannya ke konteks, sehingga jawaban lebih grounded. Ini akan dibahas di bab berikutnya.

## Peta ekosistem: provider dan model lokal

Ada dua jalur utama untuk memakai model generative AI.

### Jalur 1: Provider (hosted)
Model dijalankan di server penyedia, dan kita mengaksesnya lewat situs, aplikasi, atau API.

| Penyedia | Contoh model | Catatan |
|---|---|---|
| Google | Gemini | Akses API lewat Google AI Studio. Free tier punya batas pemakaian. |
| Anthropic | Claude | Akses lewat aplikasi Claude dan API. |
| OpenAI | ChatGPT (GPT) | Akses lewat aplikasi ChatGPT dan API. |
| Microsoft | Copilot | Terintegrasi dengan produk Microsoft. |
| Alibaba | Qwen | Ada versi hosted dan versi open-weight. |
| DeepSeek | DeepSeek | Model dari China, tersedia lewat layanan dan API. |
| Agregator | OpenRouter, Groq | Satu API key untuk banyak model, dengan batas pemakaian yang berbeda. |

**Kelebihan:** model besar, mudah dipakai, tidak perlu perangkat khusus.
**Kekurangan:** butuh internet, data dikirim ke penyedia, ada kuota atau biaya, dan versi model bisa berubah.

### Jalur 2: Model open-weight yang bisa dijalankan di laptop
Bobot model bisa diunduh dan dijalankan di komputer sendiri. Setelah diunduh, model bisa berjalan tanpa internet dan tanpa batas request.

| Model | Pembuat |
|---|---|
| Llama | Meta |
| Gemma | Google |
| Mistral | Mistral AI |
| Phi | Microsoft |
| Qwen (versi open-weight) | Alibaba |

**Alat untuk menjalankannya:**
- **Ollama:** paling mudah, memakai perintah sederhana dan punya aplikasi chat.
- **LM Studio:** antarmuka grafis, cocok untuk pemula yang tidak mau memakai terminal.
- **llama.cpp:** lebih teknis, dipakai jika ingin kontrol penuh.

**Kelebihan:** offline, data tetap di laptop, tidak ada batas request, dan versi model tetap sehingga hasil praktik bisa diulang.
**Kekurangan:** butuh perangkat yang memadai, model kecil biasanya kurang kuat dibanding model besar, dan unduhan awal bisa berukuran beberapa gigabyte.

### Perbandingan singkat

| Aspek | Provider (hosted) | Open-weight lokal |
|---|---|---|
| Internet | Wajib | Hanya saat mengunduh |
| Kekuatan model | Umumnya lebih besar | Tergantung ukuran model dan perangkat |
| Batas pemakaian | Ada, berbeda tiap layanan | Tidak ada |
| Data | Dikirim ke penyedia | Tetap di laptop |
| Versi model | Bisa berubah | Tetap sampai diunduh ulang |

**Pertanyaan diskusi:** untuk aplikasi kampus yang menyimpan data mahasiswa, jalur mana yang lebih tepat, dan mengapa?

*Nama model dan layanan berubah cepat. Cek kembali situs resmi masing-masing sebelum praktik.*

## 1. Instalasi dan koneksi

In [ ]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 25.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [ ]:
from google.colab import userdata
import os
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

from google import genai
from google.genai import types

client = genai.Client()

# Lihat model yang tersedia untuk akunmu
for m in client.models.list():
    if 'flash' in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


Pilih satu nama model dari daftar di atas. Nama model bisa berubah, jadi selalu ambil dari daftar terbaru.

In [ ]:
MODEL = 'gemini-3.1-flash-lite'  # ganti jika namanya berbeda di daftar
print('Model:', MODEL)

Model: gemini-3.1-flash-lite


## 2. Panggilan pertama dan token

Setiap respons membawa informasi jumlah token. Token adalah unit yang dihitung untuk biaya dan batas konteks. Jumlah token tidak sama dengan jumlah kata.

In [ ]:
r = client.models.generate_content(
    model=MODEL,
    contents='Jelaskan apa itu large language model dalam satu kalimat saja.'
)
print(r.text)
print('Token input :', r.usage_metadata.prompt_token_count)
print('Token output:', r.usage_metadata.candidates_token_count)

*Large Language Model* adalah model kecerdasan buatan berbasis jaringan saraf tiruan yang dilatih menggunakan data teks dalam jumlah masif untuk memahami, memproses, dan menghasilkan bahasa manusia secara natural.
Token input : 14
Token output: 40


**Coba:** ubah pertanyaan menjadi lebih panjang, lalu bandingkan jumlah token input-nya.

## 3. Pengaruh temperature

`temperature` mengatur seberapa sering model memilih kandidat kata yang kurang umum. Nilai rendah cenderung menghasilkan output yang lebih stabil. Nilai tinggi menghasilkan output yang lebih bervariasi. Temperature 0 pun tidak selalu menghasilkan teks yang persis sama.

In [ ]:
prompt = 'Tulis satu slogan pendek untuk kantin kampus.'

for t in [0.0, 1.0]:
    print(f'--- temperature = {t} ---')
    for i in range(3):
        r = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=t)
        )
        print(r.text.strip())

--- temperature = 0.0 ---
Berikut adalah beberapa pilihan slogan untuk kantin kampus, tergantung pada kesan yang ingin dibangun:

1. **Fokus pada rasa:** "Rasa Rumahan, Harga Teman."
2. **Fokus pada energi:** "Isi Energi, Raih Prestasi."
3. **Singkat & *Catchy*:** "Kantin Kampus: Kenyang, Hemat, Semangat!"
4. **Fokus pada kebersamaan:** "Tempat Makan, Ruang Pertemanan."
5. **Modern:** "Solusi Lapar di Sela Kuliah."

Pilihan terbaik: **"Rasa Rumahan, Harga Teman."** (Sangat cocok dengan target mahasiswa yang mencari kenyamanan dan harga terjangkau).
Berikut adalah beberapa pilihan slogan untuk kantin kampus, tergantung pada kesan yang ingin dibangun:

1. **Fokus pada rasa:** "Rasa Rumahan, Harga Teman."
2. **Fokus pada energi:** "Isi Energi, Raih Prestasi."
3. **Singkat & *Catchy*:** "Kantin Kampus: Kenyang, Hemat, Semangat!"
4. **Fokus pada kebersamaan:** "Tempat Makan, Ruang Pertemanan."
5. **Modern:** "Solusi Lapar di Sela Kuliah."

Pilihan terbaik: **"Rasa Rumahan, Harga Teman."** (

## 4. Instruksi sistem dan peran

Instruksi sistem (`system_instruction`) mengatur peran dan batasan model, misalnya bahwa model harus menjawab singkat dalam bahasa Indonesia. Instruksi ini dikirim bersama setiap permintaan.

In [ ]:
instruksi = ('Kamu adalah asisten informasi akademik kampus. '
             'Jawab dalam bahasa Indonesia, singkat, dan jika tidak yakin, katakan bahwa kamu tidak yakin.')

r = client.models.generate_content(
    model=MODEL,
    contents='Berapa biaya pendaftaran ulang mahasiswa baru?',
    config=types.GenerateContentConfig(system_instruction=instruksi)
)
print(r.text)

Mohon maaf, saya tidak yakin. Informasi biaya pendaftaran ulang bergantung pada kebijakan masing-masing universitas dan jalur masuk yang dipilih.

Silakan informasikan nama universitas Anda agar saya dapat membantu mencarikan informasi yang lebih akurat.


**Perhatikan:** model mungkin tidak tahu biaya yang berlaku di kampusmu. Itu contoh halusinasi yang perlu diperiksa ke sumber resmi, bukan dijawab dari tebakan model.

## 5. Output terstruktur dalam JSON

Aplikasi biasanya membutuhkan output yang bisa dibaca program. Kode di bawah meminta JSON, lalu memeriksa apakah hasilnya benar-benar bisa dibaca.

In [ ]:
import json

pengumuman = [
    'Seminar AI akan diadakan pada 12 November 2026 di Gedung Rektorat lantai 3.',
    'Pendaftaran praktikum Basis Data dibuka Senin 3 November di Lab Komputer B.',
    'Rapat himpunan mahasiswa D3 TI dijadwalkan Kamis 5 November di Ruang Sidang 2.',
]

instruksi_json = 'Ekstrak nama kegiatan, tanggal, dan lokasi dari teks berikut. Jawab hanya dalam format JSON dengan kunci nama, tanggal, dan lokasi.\n\nTeks: '

gagal = 0
for teks in pengumuman:
    r = client.models.generate_content(
        model=MODEL,
        contents=instruksi_json + teks,
        config=types.GenerateContentConfig(response_mime_type='application/json')
    )
    try:
        print(json.loads(r.text))
    except json.JSONDecodeError:
        gagal += 1
        print('FORMAT GAGAL:', r.text)

print('Format gagal:', gagal, 'dari', len(pengumuman))

{'nama': 'Seminar AI', 'tanggal': '12 November 2026', 'lokasi': 'Gedung Rektorat lantai 3'}
{'nama': 'Pendaftaran praktikum Basis Data', 'tanggal': 'Senin 3 November', 'lokasi': 'Lab Komputer B'}
{'nama': 'Rapat himpunan mahasiswa D3 TI', 'tanggal': 'Kamis 5 November', 'lokasi': 'Ruang Sidang 2'}
Format gagal: 0 dari 3


## 6. Riwayat percakapan (chatbot sederhana)

Model tidak mengingat percakapan sebelumnya secara otomatis. Agar terasa nyambung, setiap giliran harus mengirim ulang riwayat pesan. Semakin panjang riwayat, semakin besar jumlah token input.

In [ ]:
riwayat = []

def tanya(pesan):
    riwayat.append(types.Content(role='user', parts=[types.Part(text=pesan)]))
    r = client.models.generate_content(model=MODEL, contents=riwayat)
    riwayat.append(types.Content(role='model', parts=[types.Part(text=r.text)]))
    print('Kamu  :', pesan)
    print('Model :', r.text.strip())
    print('(token input:', r.usage_metadata.prompt_token_count, ')')
    print()

tanya('Saya mahasiswa D3 Teknologi Informasi. Ingat ya.')
tanya('Saya sedang belajar apa?')

Kamu  : Saya mahasiswa D3 Teknologi Informasi. Ingat ya.
Model : Siap, dicatat! Sebagai mahasiswa **D3 Teknologi Informasi**, saya paham bahwa fokus kamu kemungkinan besar ada pada **praktik, implementasi teknologi, dan pemecahan masalah teknis** yang aplikatif.

Karena ini jenjang vokasi (D3), biasanya mata kuliah kamu akan sangat padat dengan praktikum seperti:
*   **Pemrograman** (Web, Mobile, atau Desktop).
*   **Basis Data** (SQL, perancangan database).
*   **Jaringan Komputer & Administrasi Sistem**.
*   **Analisis Sistem & Rekayasa Perangkat Lunak**.
*   **Tugas Akhir (TA)** yang biasanya berupa pembuatan aplikasi atau infrastruktur TI.

Ada yang bisa saya bantu terkait kuliah kamu? Misalnya:
1.  **Diskusi materi kuliah** yang sulit dipahami (algoritma, struktur data, networking, dll).
2.  **Bantuan pengerjaan coding** (debugging, struktur database, atau logic pemrograman).
3.  **Ide atau konsultasi Tugas Akhir (TA)**.
4.  **Persiapan magang atau sertifikasi profesi**.
5.  **Tip

**Coba:** jalankan ulang sel di atas tanpa sel pembuatan `riwayat = []`, lalu bandingkan jawabannya. Perhatikan juga bagaimana jumlah token input bertambah pada setiap giliran.

## 7. Mini-evaluasi

Output model bersifat terbuka, sehingga tidak bisa dinilai hanya dengan accuracy. Beri skor manual untuk setiap kasus berikut dengan skala 1 sampai 3 pada ketepatan, kepatuhan format, dan konsistensi.

In [ ]:
kasus = [
    'Ringkas dalam tiga poin: Universitas Brawijaya adalah perguruan tinggi negeri yang berlokasi di Malang, Jawa Timur.',
    'Ubah kalimat ini menjadi bahasa santai tanpa mengubah maknanya: Pendaftaran wajib dilakukan sebelum batas waktu.',
    'Kapan Universitas Brawijaya didirikan? Jawab singkat dan sebutkan seberapa yakin kamu.',
    'Klasifikasikan sebagai Positif, Negatif, atau Netral: Pelayanan perpustakaan sangat membantu.',
    'Buat tabel markdown dengan kolom Nama dan Umur untuk tiga orang fiktif.',
]

for i, k in enumerate(kasus, 1):
    r = client.models.generate_content(model=MODEL, contents=k)
    print(f'--- Kasus {i} ---')
    print(r.text)
    print()

--- Kasus 1 ---
Berikut adalah ringkasan dalam tiga poin:

1. **Status:** Merupakan perguruan tinggi negeri.
2. **Lokasi:** Terletak di kota Malang, Jawa Timur.
3. **Identitas:** Dikenal dengan nama Universitas Brawijaya.

--- Kasus 2 ---
Berikut adalah beberapa pilihan kalimat santai yang bisa kamu gunakan, tergantung situasinya:

*   **Paling umum:** "Jangan lupa daftar sebelum deadline-nya ya."
*   **Lebih santai:** "Buruan daftar sebelum ditutup."
*   **Singkat:** "Daftar sekarang sebelum kehabisan waktu."
*   **Sedikit akrab:** "Pastikan sudah daftar sebelum waktunya habis, ya!"
*   **To the point:** "Jangan telat daftarnya, nanti keburu ditutup."

--- Kasus 3 ---
Universitas Brawijaya didirikan pada **5 Januari 1963**.

Tingkat keyakinan: **100%**.

--- Kasus 4 ---
Klasifikasi: **Positif**

--- Kasus 5 ---
Berikut adalah tabel dengan tiga orang fiktif:

| Nama | Umur |
| :--- | :--- |
| Budi Santoso | 25 |
| Siti Aminah | 30 |
| Andi Wijaya | 22 |



## Refleksi

1. Pada kasus faktual, bagaimana kamu memastikan jawaban model benar?
2. Untuk tugas apa temperature rendah lebih cocok, dan mengapa?
3. Apa dampak riwayat percakapan yang panjang terhadap biaya dan kecepatan respons?
4. Jika aplikasi kampus dipakai mahasiswa, data apa yang tidak boleh dikirim ke layanan ini?

**Catatan keamanan:** free tier Gemini di luar EU, UK, dan EEA dapat memakai data untuk pelatihan model. Jangan memasukkan data pribadi, nilai, atau dokumen internal kampus. Setelah praktik, hapus API key di AI Studio jika tidak dipakai lagi.